# Addresses26 — Free Automated Agent Discovery v3 High-Yield Pilot

Purpose: improve the free estate-agent discovery pipeline by reducing low-value search noise and increasing property-listing signal.

This notebook:
- loads verified Land Registry BN transactions for 2021–2025
- builds a balanced pilot sample
- creates targeted searches for property portals and local agents
- uses free `ddgs` / `duckduckgo_search` if available
- filters low-value postcode-info domains
- boosts likely listing/agent sources
- extracts candidate agents using a larger editable dictionary plus phrase matching
- builds candidate, canonical agent, and transaction-agent tables

Important: HM Land Registry Price Paid Data does **not** contain estate-agent data. Agent attribution here is probabilistic and must remain separate from verified transaction data.


In [1]:
# ============================================================
# 0. Optional install: free web-search package
# ============================================================
import sys, subprocess, importlib.util

for pkg in ["ddgs", "duckduckgo_search"]:
    if importlib.util.find_spec(pkg) is None:
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
        except Exception as e:
            print(f"Could not install {pkg}: {e}")


In [2]:
# ============================================================
# 1. Mount Drive and configure paths
# ============================================================
from google.colab import drive
from pathlib import Path
import pandas as pd
import numpy as np
import re, json, time, hashlib
from datetime import datetime, timezone

try:
    drive.mount('/content/drive')
except Exception as e:
    print('Drive mount warning:', e)

BASE = Path('/content/drive/MyDrive/Addresses26_Data')
DERIVED = BASE / 'derived' / 'land_registry_price_paid'
OUT = BASE / 'reports' / 'agent_attribution_free_v3_high_yield'
OUT.mkdir(parents=True, exist_ok=True)

POSTCODE_PREFIX = 'BN'
YEARS = [2021, 2022, 2023, 2024, 2025]
MAX_TRANSACTIONS_THIS_RUN = 25   # increase to 50/100 after reviewing hit rate
MAX_RESULTS_PER_QUERY = 8
SLEEP_SECONDS_BETWEEN_QUERIES = 8
RANDOM_SEED = 26

# v3 controls
PORTAL_ONLY_MODE = False  # True = only portal/domain-targeted queries; False = portal + general
KEEP_ONLY_USEFUL_DOMAINS = False # True may improve precision but lower recall

print('BASE:', BASE)
print('DERIVED:', DERIVED)
print('OUT:', OUT)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Addresses26_Data
DERIVED: /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid
OUT: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution_free_v3_high_yield


In [3]:
# ============================================================
# 2. Load verified yearly Land Registry parquet files
# ============================================================
frames = []
missing_files = []
for y in YEARS:
    p = DERIVED / f'pp_bn_standard_{y}.parquet'
    if p.exists():
        df = pd.read_parquet(p)
        df.columns = [str(c).strip().lower() for c in df.columns]
        df['sale_year_source_file'] = y
        frames.append(df)
        print(f'Loaded {y}: {len(df):,} rows')
    else:
        missing_files.append(str(p))

if not frames:
    raise FileNotFoundError('No yearly parquet files found. Check DERIVED path and filenames.')

sales = pd.concat(frames, ignore_index=True)
print('Combined rows:', len(sales))
if missing_files:
    print('Missing files:', missing_files)


Loaded 2021: 18,279 rows
Loaded 2022: 14,934 rows
Loaded 2023: 11,515 rows
Loaded 2024: 12,532 rows
Loaded 2025: 11,425 rows
Combined rows: 68685


In [4]:
# ============================================================
# 3. Cleaning helpers and transaction preparation
# ============================================================
for c in ['paon', 'saon', 'street', 'locality', 'town_city', 'district', 'county', 'postcode', 'price', 'date_of_transfer']:
    if c not in sales.columns:
        sales[c] = pd.NA

def clean_token(x):
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s in {'', '<NA>', '<na>', 'NA', 'N/A', 'nan', 'NaN', 'None', 'NONE', 'null', 'NULL'}:
        return ''
    s = re.sub(r'<\s*NA\s*>', '', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip(' ,')
    return s

def clean_address_from_row(row):
    parts = [
        clean_token(row.get('saon')),
        clean_token(row.get('paon')),
        clean_token(row.get('street')),
        clean_token(row.get('locality')),
        clean_token(row.get('town_city')),
    ]
    parts = [p for p in parts if p]
    out = []
    for p in parts:
        if not out or p.upper() != out[-1].upper():
            out.append(p)
    return ' '.join(out)

def street_name_from_row(row):
    return clean_token(row.get('street'))

def make_transaction_id(row):
    for c in ['transaction_unique_identifier', 'transaction_id', 'uid']:
        v = clean_token(row.get(c))
        if v:
            return v
    base = '|'.join([
        clean_token(row.get('price')),
        clean_token(row.get('date_of_transfer')),
        clean_token(row.get('postcode')),
        clean_address_from_row(row),
    ])
    return hashlib.sha1(base.encode('utf-8')).hexdigest()[:16]

sales['address_clean'] = sales.apply(clean_address_from_row, axis=1)
sales['street_clean'] = sales.apply(street_name_from_row, axis=1)
sales['postcode_clean'] = sales['postcode'].astype(str).str.upper().str.strip()
sales['town_clean'] = sales['town_city'].apply(clean_token)
sales['sale_date'] = pd.to_datetime(sales['date_of_transfer'], errors='coerce')
sales['sale_year'] = sales['sale_date'].dt.year.fillna(sales['sale_year_source_file']).astype('Int64')
sales['transaction_id'] = sales.apply(make_transaction_id, axis=1)
sales['postcode_district'] = sales['postcode_clean'].str.extract(r'^([A-Z]{1,2}\d{1,2}[A-Z]?)', expand=False)

sales = sales[(sales['postcode_clean'].str.startswith(POSTCODE_PREFIX, na=False)) & (sales['address_clean'].str.len() > 3)].copy()
print('Usable sales rows:', len(sales))
display(sales[['transaction_id','sale_year','address_clean','street_clean','postcode_clean','price','sale_date']].head(10))


Usable sales rows: 68685


,transaction_id,sale_year,address_clean,street_clean,postcode_clean,price,sale_date
0,{F87E72F8-D9D5-176C-E053-6B04A8C0D2BE},2021,58 PORTLAND ROAD HOVE,PORTLAND ROAD,BN3 5DL,650000,2021-01-03
1,{BC8936BB-74DB-0E2C-E053-6C04A8C0DBF4},2021,SECOND FLOOR FLAT 1 WEST HILL ROAD BRIGHTON,WEST HILL ROAD,BN1 3RT,265000,2021-01-04
2,{BC8936BB-F58B-0E2C-E053-6C04A8C0DBF4},2021,45 GLENVILLE ROAD RUSTINGTON LITTLEHAMPTON,GLENVILLE ROAD,BN16 2EA,505000,2021-01-04
3,{BC8936BB-743D-0E2C-E053-6C04A8C0DBF4},2021,"FLAT 1 THE LEAS, 34 - 35 SUSSEX SQUARE BRIGHTON",SUSSEX SQUARE,BN2 5AD,430000,2021-01-04
4,{BC8936BB-6FA6-0E2C-E053-6C04A8C0DBF4},2021,25 OLD CAMP ROAD EASTBOURNE,OLD CAMP ROAD,BN20 8DL,800000,2021-01-04
5,{BC8936BB-F642-0E2C-E053-6C04A8C0DBF4},2021,4 HAWTH GROVE SEAFORD,HAWTH GROVE,BN25 2RP,370000,2021-01-04
6,{BEF7EBBF-7F92-7A76-E053-6B04A8C092F7},2021,43A SOUTHDOWN ROAD SHOREHAM-BY-SEA,SOUTHDOWN ROAD,BN43 5AL,242500,2021-01-04
7,{BEF7EBBF-807E-7A76-E053-6B04A8C092F7},2021,40 PENLANDS VALE STEYNING,PENLANDS VALE,BN44 3PL,441200,2021-01-04
8,{BC8936BB-72FA-0E2C-E053-6C04A8C0DBF4},2021,FLAT 12 48 - 50 DYKE ROAD BRIGHTON,DYKE ROAD,BN1 3JB,230000,2021-01-05
9,{BEF7EBBF-4E0E-7A76-E053-6B04A8C092F7},2021,102 ELDRED AVENUE BRIGHTON,ELDRED AVENUE,BN1 5EH,497500,2021-01-05


In [5]:
# ============================================================
# 4. Balanced pilot sample
# ============================================================
rng = np.random.default_rng(RANDOM_SEED)
sample_source = sales.copy()
sample_source['_rand'] = rng.random(len(sample_source))

# v3: spread across years first, then district, but avoid taking only one district
samples = []
per_year = max(1, MAX_TRANSACTIONS_THIS_RUN // len(YEARS))
for y, gy in sample_source.groupby('sale_year'):
    if int(y) not in YEARS:
        continue
    # one per district before extras
    district_first = gy.sort_values('_rand').groupby('postcode_district', dropna=False).head(1)
    take = district_first.sort_values('_rand').head(per_year)
    samples.append(take)

pilot = pd.concat(samples, ignore_index=True) if samples else sample_source.sample(min(MAX_TRANSACTIONS_THIS_RUN, len(sample_source)), random_state=RANDOM_SEED)
if len(pilot) < MAX_TRANSACTIONS_THIS_RUN:
    used = set(pilot['transaction_id'])
    topup = sample_source[~sample_source['transaction_id'].isin(used)].sort_values('_rand').head(MAX_TRANSACTIONS_THIS_RUN - len(pilot))
    pilot = pd.concat([pilot, topup], ignore_index=True)

pilot = pilot.head(MAX_TRANSACTIONS_THIS_RUN).copy()
print('Transactions to search this run:', len(pilot))
display(pilot.groupby('sale_year').size().reset_index(name='rows'))
display(pilot.groupby('postcode_district').size().reset_index(name='rows').sort_values('rows', ascending=False).head(20))
display(pilot[['transaction_id','sale_year','postcode_district','address_clean','postcode_clean','price']].head(20))
pilot.to_csv(OUT / 'search_transaction_sample.csv', index=False)


Transactions to search this run: 25


,sale_year,rows
0,2021,5
1,2022,5
2,2023,5
3,2024,5
4,2025,5


,postcode_district,rows
14,BN3,3
6,BN2,3
7,BN20,2
0,BN1,2
5,BN18,2
1,BN11,1
4,BN16,1
3,BN13,1
8,BN21,1
2,BN12,1


,transaction_id,sale_year,postcode_district,address_clean,postcode_clean,price
0,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,225000
1,{C3C3F9B5-9D8A-362B-E053-6B04A8C03ACC},2021,BN20,126 CHERRY GARDEN ROAD EASTBOURNE,BN20 8HG,376000
2,{C8A3A576-0FD8-0425-E053-6C04A8C0947B},2021,BN21,121 GREEN STREET EASTBOURNE,BN21 1RS,338000
3,{F3B6C199-0687-6E40-E053-6C04A8C0B3B4},2021,BN6,36 OCKENDEN WAY HASSOCKS,BN6 8HS,550000
4,{CD5A9DCC-5467-310A-E053-6C04A8C00A1F},2021,BN2,53 CARLYLE AVENUE BRIGHTON,BN2 4DR,362000
5,{E2D14905-58FE-4C2D-E053-6B04A8C0422B},2022,BN3,FLAT B 5 NEW CHURCH ROAD HOVE,BN3 4AA,720000
6,{DE2D0CE0-2A0D-51EE-E053-6C04A8C00671},2022,BN13,1 TAVY CLOSE WORTHING,BN13 3PA,360000
7,{F16F63C5-9B42-0491-E053-6C04A8C032ED},2022,BN20,"APARTMENT 2 HOLYWELL COURT, 30 KING EDWARDS PA...",BN20 7FF,500000
8,{EA3278AA-7D26-2676-E053-6B04A8C015F8},2022,BN1,27 LARKFIELD WAY BRIGHTON,BN1 8EG,700000
9,{09266DDB-B494-AF90-E063-4704A8C02087},2022,BN22,1A WARTLING ROAD EASTBOURNE,BN22 7PG,255000


In [6]:
# ============================================================
# 5. High-yield domain lists and agent dictionary
# ============================================================
GOOD_DOMAINS = [
    'rightmove.co.uk', 'zoopla.co.uk', 'onthemarket.com', 'primelocation.com',
    'boomin.com', 'nethouseprices.com', 'home.co.uk', 'mouseprice.com',
    'themovemarket.com', 'propertyheads.com', 'placebuzz.com', 'propertypal.com',
    'vebra.com', 'vebraalto.com', 'media.onthemarket.com'
]

NOISE_DOMAINS = [
    'streetcheck.co.uk', 'streetlist.co.uk', 'geopunk.co.uk', 'checkmypostcode.uk',
    'postcodebyaddress.co.uk', 'findmyaddress.co.uk', 'getthedata.com',
    'crystalroof.co.uk', 'doogal.co.uk', 'ukpostcodecheck.com',
    'housesforsaletorent.co.uk'
]

KNOWN_AGENTS = sorted(set([
    'Fox & Sons', 'Connells', 'Cubitt & West', 'King & Chasemore', 'Hamptons', 'Savills',
    'Mishon Mackay', 'Mishons', 'Oakley Property', 'Brand Vaughan', 'Leaders', 'Jacobs Steel',
    'Robert Luff', 'Michael Jones', 'Sawyers', 'Austin Gray', 'Phillips and Still',
    'Kendrick Property Services', 'Coapt', 'John Hilton', 'Goldin Lemcke', 'Move Revolution',
    'Purplebricks', 'Yopa', 'Strike', 'Fine & Country', 'Jackson-Stops', 'Your Move',
    'Mansell McTaggart', 'Freeman Forman', 'Taylor Robinson', 'Choices', 'Martin & Co',
    'Belvoir', 'Hunters', 'Open House', 'Priors', 'Pearson Keehan', 'Lextons',
    'David Maslen', 'Maslen Estate Agents', 'Wheelers Estate Agents', 'Wheeler’s Estate Agents',
    'Spencer & Leigh', 'Clifford Sales & Lettings', 'Paul Bott & Company', 'Nicholas James',
    'Elliotts', 'Foster & Co', 'Healy and Newsom', 'Hyman Hill', 'Warwick Baker',
    'Bacon and Company', 'Matthew Anthony', 'Aspire Residential', 'James & James',
    'Stafford Johnson', 'Coast & Country', 'Rowland Gorringe', 'Lampons', 'Phillip Mann',
    'Charles Cox', 'Home Sweet Home', 'Town Property', 'Taylor Engley', 'Rager & Roberts',
    'Reid & Dean', 'Ancells Estates', 'Open House Estate Agents', 'Phillip Mann Estate Agents',
    'Avard Estate Agents', 'Beaumonts', 'Beaumonts Residential', 'Brighton Accommodation Agency',
    'Q Estate Agents', 'My Sales', 'HW Estate Agents', 'Lextons Brighton', 'Justin Lloyd',
    'Aston Vaughan', 'Aston & Co', 'Khalil Properties', 'Stanfords Estate Agents',
    'John Hoole', 'Wilkins Vardy', 'Callaways', 'Foster & Co Estate Agents',
    'Clifford Dann', 'Lewes Estates', 'Oakfield', 'Andrews Estate Agents',
    'Graham Butt', 'Glyn-Jones', 'Brock Taylor', 'Henry Adams', 'PSP Homes',
    'Marcus Grimes', 'Arington', 'Batten & Co', 'Bexhill Estates', 'New Foundations',
    'Abbott & Abbott', 'Rush Witt & Wilson', 'Campbell’s', 'Campbells', 'Made',
    'Wyatt Hughes', 'Oakley', 'Mishon Welton', 'Winkworth', 'Knight Frank',
]))

AGENT_PATTERNS = [
    r"(?i)marketed by[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)estate agent[s]?[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)sold by[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)listed by[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
    r"(?i)agent[:\s]+([A-Za-z0-9&.,'’\- ]{2,80})",
]

print('Known agents:', len(KNOWN_AGENTS))


Known agents: 111


In [7]:
# ============================================================
# 6. Query builder v3 — property-portal targeted
# ============================================================
def simplify_address_for_query(address):
    s = clean_token(address)
    # Remove flat descriptors for a broader fallback; keep original elsewhere.
    s2 = re.sub(r'\b(FLAT|APARTMENT|MAISONETTE|GROUND FLOOR FLAT|LOWER GROUND FLOOR FLAT|FIRST FLOOR FLAT|SECOND FLOOR FLAT)\b', '', s, flags=re.I)
    s2 = re.sub(r'\s+', ' ', s2).strip()
    return s2 or s

def build_queries_v3(row):
    address = clean_token(row.get('address_clean'))
    broad_address = simplify_address_for_query(address)
    postcode = clean_token(row.get('postcode_clean'))
    town = clean_token(row.get('town_clean'))
    street = clean_token(row.get('street_clean'))
    queries = []

    if address and postcode:
        # Highest-yield portal-targeted searches
        queries.append(f'"{address}" "{postcode}" rightmove')
        queries.append(f'"{address}" "{postcode}" zoopla')
        queries.append(f'"{address}" "{postcode}" onthemarket')
        queries.append(f'"{address}" "{postcode}" "marketed by"')

    if broad_address and broad_address != address and postcode:
        queries.append(f'"{broad_address}" "{postcode}" rightmove')
        queries.append(f'"{broad_address}" "{postcode}" zoopla')

    if street and postcode:
        queries.append(f'"{street}" "{postcode}" "sold house prices"')
        queries.append(f'"{street}" "{postcode}" "property for sale"')

    if postcode and town:
        queries.append(f'"{postcode}" "{town}" "rightmove" "sold"')
        queries.append(f'"{postcode}" "{town}" "estate agent"')

    if not PORTAL_ONLY_MODE and address and postcode:
        queries.append(f'"{address}" "{postcode}" estate agent')
        queries.append(f'"{address}" "{postcode}" property for sale')

    # Deduplicate and remove accidental NA fragments
    out, seen = [], set()
    for q in queries:
        q = re.sub(r'\s+', ' ', q).replace('<NA>', '').strip()
        if q and q.lower() not in seen:
            seen.add(q.lower())
            out.append(q)
    return out[:8]  # cap queries per transaction

pilot['queries'] = pilot.apply(build_queries_v3, axis=1)
for _, row in pilot.head(3).iterrows():
    print(row['transaction_id'], row['address_clean'], row['postcode_clean'])
    for q in row['queries']:
        print(' query:', q)


{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING BN11 3QW
 query: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" rightmove
 query: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" zoopla
 query: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" onthemarket
 query: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" "marketed by"
 query: "9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" rightmove
 query: "9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" zoopla
 query: "ST VALERIE ROAD" "BN11 3QW" "sold house prices"
 query: "ST VALERIE ROAD" "BN11 3QW" "property for sale"
{C3C3F9B5-9D8A-362B-E053-6B04A8C03ACC} 126 CHERRY GARDEN ROAD EASTBOURNE BN20 8HG
 query: "126 CHERRY GARDEN ROAD EASTBOURNE" "BN20 8HG" rightmove
 query: "126 CHERRY GARDEN ROAD EASTBOURNE" "BN20 8HG" zoopla
 query: "126 CHERRY GARDEN ROAD EASTBOURNE" "BN20 8HG" onthemarket
 query: "126 CHERRY GARDEN ROAD EASTBOURNE"

In [8]:
# ============================================================
# 7. Free search adapter and query plan
# ============================================================
SEARCH_BACKEND = None
try:
    from ddgs import DDGS
    SEARCH_BACKEND = 'ddgs'
except Exception:
    try:
        from duckduckgo_search import DDGS
        SEARCH_BACKEND = 'duckduckgo_search'
    except Exception as e:
        print('No free search backend available:', e)
        SEARCH_BACKEND = None

query_plan_rows = []
for _, row in pilot.iterrows():
    for q in row['queries']:
        query_plan_rows.append({
            'transaction_id': row['transaction_id'],
            'sale_year': int(row['sale_year']) if not pd.isna(row['sale_year']) else None,
            'postcode_district': row['postcode_district'],
            'address_clean': row['address_clean'],
            'postcode': row['postcode_clean'],
            'street': row['street_clean'],
            'price': row.get('price'),
            'query': q,
        })
query_plan = pd.DataFrame(query_plan_rows)
query_plan.to_csv(OUT / 'agent_discovery_query_plan_v3.csv', index=False)
print('SEARCH_BACKEND:', SEARCH_BACKEND)
print('Query plan rows:', len(query_plan))
display(query_plan.head(20))


SEARCH_BACKEND: ddgs
Query plan rows: 200


,transaction_id,sale_year,postcode_district,address_clean,postcode,street,price,query
0,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH..."
1,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH..."
2,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH..."
3,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH..."
4,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""9 WILMINGTON COURT ST VALERIE ROAD WORTHING"" ..."
5,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""9 WILMINGTON COURT ST VALERIE ROAD WORTHING"" ..."
6,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""ST VALERIE ROAD"" ""BN11 3QW"" ""sold house prices"""
7,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},2021,BN11,FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING,BN11 3QW,ST VALERIE ROAD,225000,"""ST VALERIE ROAD"" ""BN11 3QW"" ""property for sale"""
8,{C3C3F9B5-9D8A-362B-E053-6B04A8C03ACC},2021,BN20,126 CHERRY GARDEN ROAD EASTBOURNE,BN20 8HG,CHERRY GARDEN ROAD,376000,"""126 CHERRY GARDEN ROAD EASTBOURNE"" ""BN20 8HG""..."
9,{C3C3F9B5-9D8A-362B-E053-6B04A8C03ACC},2021,BN20,126 CHERRY GARDEN ROAD EASTBOURNE,BN20 8HG,CHERRY GARDEN ROAD,376000,"""126 CHERRY GARDEN ROAD EASTBOURNE"" ""BN20 8HG""..."


In [9]:
# ============================================================
# 8. Run slow free search
# ============================================================
raw_results = []
if SEARCH_BACKEND is None:
    print('No search backend available. Query plan saved only.')
else:
    with DDGS() as ddgs:
        total = len(query_plan)
        for idx, r in query_plan.iterrows():
            print(f"[{idx+1}/{total}] {r['transaction_id']} :: {r['query']}")
            try:
                try:
                    results = list(ddgs.text(r['query'], region='uk-en', max_results=MAX_RESULTS_PER_QUERY))
                except TypeError:
                    results = list(ddgs.text(r['query'], max_results=MAX_RESULTS_PER_QUERY))
                for rank, item in enumerate(results, start=1):
                    raw_results.append({
                        'transaction_id': r['transaction_id'],
                        'query': r['query'],
                        'rank': rank,
                        'title': item.get('title') or item.get('heading') or '',
                        'href': item.get('href') or item.get('url') or '',
                        'body': item.get('body') or item.get('snippet') or '',
                        'searched_utc': datetime.now(timezone.utc).isoformat(),
                    })
            except Exception as e:
                raw_results.append({
                    'transaction_id': r['transaction_id'],
                    'query': r['query'],
                    'rank': None,
                    'title': '',
                    'href': '',
                    'body': '',
                    'error': str(e),
                    'searched_utc': datetime.now(timezone.utc).isoformat(),
                })
            time.sleep(SLEEP_SECONDS_BETWEEN_QUERIES)

raw = pd.DataFrame(raw_results)
raw_path = OUT / 'agent_discovery_raw_search_results_v3.csv'
raw.to_csv(raw_path, index=False)
print('Saved raw results:', raw_path, 'rows:', len(raw))
display(raw.head(20))


[1/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" rightmove
[2/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" zoopla
[3/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" onthemarket
[4/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" "marketed by"
[5/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" rightmove
[6/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "9 WILMINGTON COURT ST VALERIE ROAD WORTHING" "BN11 3QW" zoopla
[7/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "ST VALERIE ROAD" "BN11 3QW" "sold house prices"
[8/200] {C3C3F9B6-83F2-362B-E053-6B04A8C03ACC} :: "ST VALERIE ROAD" "BN11 3QW" "property for sale"
[9/200] {C3C3F9B5-9D8A-362B-E053-6B04A8C03ACC} :: "126 CHERRY GARDEN ROAD EASTBO

,transaction_id,query,rank,title,href,body,searched_utc,error
0,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",1.0,"Flat 3, Wilmington Court, St. Valerie Road, Wo...",https://www.ukpostcodecheck.com/address/73b781...,This address belongs to Flat 3 located in the ...,2026-04-27T18:42:32.716568+00:00,NaN
1,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",2.0,"Houses for sale & to rent in BN11 3QW, St Vale...",https://housesforsaletorent.co.uk/houses/engla...,Properties in BN11 3QW have an average house p...,2026-04-27T18:42:32.716607+00:00,NaN
2,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",3.0,"House Prices in St Valerie Road, Worthing, Wes...",https://www.rightmove.co.uk/house-prices/bn11/...,"December 26, 2021 - The average price for a pr...",2026-04-27T18:42:32.716620+00:00,NaN
3,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",4.0,"St Valerie Road, Worthing, BN11 3QW - Resident...",https://www.192.com/places/bn/bn11-3/bn11-3qw/,"Who lives in St Valerie Road, Worthing, BN11 3...",2026-04-27T18:42:32.716631+00:00,NaN
4,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",5.0,BN11 3QN - Check My Postcode,https://checkmypostcode.uk/bn113qn,BN11 3QN is in the Worthing travel to work area.,2026-04-27T18:42:32.716641+00:00,NaN
5,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",6.0,"House Prices in Bath Road, Worthing, West Suss...",https://www.rightmove.co.uk/house-prices/bn11/...,"The average price for a property in Bath Road,...",2026-04-27T18:42:32.716651+00:00,NaN
6,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",7.0,Properties To Rent in Worthing | Rightmove,https://www.rightmove.co.uk/property-to-rent/W...,2 weeks ago - Flats & Houses To Rent in Worthi...,2026-04-27T18:42:32.716661+00:00,NaN
7,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",8.0,Properties For Sale in Worthing | Rightmove,https://www.rightmove.co.uk/property-for-sale/...,Flats & Houses For Sale in Worthing - Find pro...,2026-04-27T18:42:32.716670+00:00,NaN
8,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",1.0,Sold House Prices - Get UK house prices online...,https://www.zoopla.co.uk/property/flat-4/wilmi...,Use Zoopla to discover the latest sold house p...,2026-04-27T18:42:44.875739+00:00,NaN
9,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",2.0,"Flat 3, Wilmington Court, St. Valerie Road, Wo...",https://www.ukpostcodecheck.com/address/73b781...,This address belongs to Flat 3 located in the ...,2026-04-27T18:42:44.875768+00:00,NaN


In [10]:
# ============================================================
# 9. Domain classification and optional filtering
# ============================================================
def domain_from_url(url):
    s = clean_token(url).lower()
    m = re.search(r'https?://(?:www\.)?([^/]+)', s)
    return m.group(1) if m else ''

def classify_domain(url):
    d = domain_from_url(url)
    if any(g in d for g in GOOD_DOMAINS):
        return 'good_property_domain'
    if any(n in d for n in NOISE_DOMAINS):
        return 'noise_domain'
    if any(x in d for x in ['property', 'estate', 'agent', 'homes', 'home', 'house']):
        return 'possible_property_domain'
    return 'general_domain'

if 'raw' in globals() and len(raw):
    raw['domain'] = raw['href'].apply(domain_from_url)
    raw['domain_class'] = raw['href'].apply(classify_domain)
    raw.to_csv(OUT / 'agent_discovery_raw_search_results_v3_classified.csv', index=False)
    print(raw['domain_class'].value_counts(dropna=False))
    display(raw[['domain','domain_class','title','href','body']].head(30))
    if KEEP_ONLY_USEFUL_DOMAINS:
        raw_for_extraction = raw[raw['domain_class'].isin(['good_property_domain','possible_property_domain'])].copy()
    else:
        raw_for_extraction = raw.copy()
else:
    raw_for_extraction = pd.DataFrame()


domain_class
general_domain              206
good_property_domain         49
noise_domain                 28
possible_property_domain     25
Name: count, dtype: int64


,domain,domain_class,title,href,body
0,ukpostcodecheck.com,noise_domain,"Flat 3, Wilmington Court, St. Valerie Road, Wo...",https://www.ukpostcodecheck.com/address/73b781...,This address belongs to Flat 3 located in the ...
1,housesforsaletorent.co.uk,noise_domain,"Houses for sale & to rent in BN11 3QW, St Vale...",https://housesforsaletorent.co.uk/houses/engla...,Properties in BN11 3QW have an average house p...
2,rightmove.co.uk,good_property_domain,"House Prices in St Valerie Road, Worthing, Wes...",https://www.rightmove.co.uk/house-prices/bn11/...,"December 26, 2021 - The average price for a pr..."
3,192.com,general_domain,"St Valerie Road, Worthing, BN11 3QW - Resident...",https://www.192.com/places/bn/bn11-3/bn11-3qw/,"Who lives in St Valerie Road, Worthing, BN11 3..."
4,checkmypostcode.uk,noise_domain,BN11 3QN - Check My Postcode,https://checkmypostcode.uk/bn113qn,BN11 3QN is in the Worthing travel to work area.
5,rightmove.co.uk,good_property_domain,"House Prices in Bath Road, Worthing, West Suss...",https://www.rightmove.co.uk/house-prices/bn11/...,"The average price for a property in Bath Road,..."
6,rightmove.co.uk,good_property_domain,Properties To Rent in Worthing | Rightmove,https://www.rightmove.co.uk/property-to-rent/W...,2 weeks ago - Flats & Houses To Rent in Worthi...
7,rightmove.co.uk,good_property_domain,Properties For Sale in Worthing | Rightmove,https://www.rightmove.co.uk/property-for-sale/...,Flats & Houses For Sale in Worthing - Find pro...
8,zoopla.co.uk,good_property_domain,Sold House Prices - Get UK house prices online...,https://www.zoopla.co.uk/property/flat-4/wilmi...,Use Zoopla to discover the latest sold house p...
9,ukpostcodecheck.com,noise_domain,"Flat 3, Wilmington Court, St. Valerie Road, Wo...",https://www.ukpostcodecheck.com/address/73b781...,This address belongs to Flat 3 located in the ...


In [11]:
# ============================================================
# 10. Extract candidate agents and score confidence
# ============================================================
def normalise_agent_name(name):
    s = clean_token(name)
    s = re.sub(r'\s+', ' ', s).strip(' -–—|,.;:')
    # Stop at common trailing junk
    s = re.split(r'\b(?:for sale|sold price|property|rightmove|zoopla|onthemarket|details|branch|valuation|lettings|estate agents?)\b', s, flags=re.I)[0]
    s = re.sub(r'\s+', ' ', s).strip(' -–—|,.;:')
    return s[:120]

def extract_agent_candidates(text):
    text = clean_token(text)
    if not text:
        return []
    found = []
    low = text.lower()
    for agent in KNOWN_AGENTS:
        if agent.lower() in low:
            found.append({'agent_name': agent, 'method': 'known_agent_dictionary'})
    for pat in AGENT_PATTERNS:
        for m in re.finditer(pat, text):
            candidate = normalise_agent_name(m.group(1))
            if candidate and len(candidate) >= 3 and not re.search(r'^(the|and|for|sale|house|property)$', candidate, flags=re.I):
                found.append({'agent_name': candidate, 'method': 'phrase_regex'})
    dedup = {}
    for x in found:
        k = x['agent_name'].lower()
        if k not in dedup:
            dedup[k] = x
    return list(dedup.values())

def score_candidate(row, agent_method):
    score = 0
    title = clean_token(row.get('title'))
    body = clean_token(row.get('body'))
    url = clean_token(row.get('href'))
    query = clean_token(row.get('query'))
    combined = f'{title} {body} {query}'.lower()
    domain_class = clean_token(row.get('domain_class'))

    if agent_method == 'known_agent_dictionary':
        score += 35
    if agent_method == 'phrase_regex':
        score += 25
    if 'marketed by' in combined:
        score += 25
    if 'sold by' in combined:
        score += 20
    if 'estate agent' in combined:
        score += 10
    if 'property for sale' in combined or 'for sale' in combined:
        score += 8
    if domain_class == 'good_property_domain':
        score += 25
    elif domain_class == 'possible_property_domain':
        score += 10
    elif domain_class == 'noise_domain':
        score -= 15
    if re.search(r'BN\d', combined, flags=re.I):
        score += 5
    try:
        if int(row.get('rank')) in [1, 2]:
            score += 5
    except Exception:
        pass
    return max(0, min(score, 100))

def confidence_label(score):
    if score >= 75:
        return 'high_candidate'
    if score >= 50:
        return 'medium_candidate'
    if score >= 30:
        return 'low_candidate'
    return 'weak_signal'

candidate_rows = []
if len(raw_for_extraction):
    for _, rr in raw_for_extraction.iterrows():
        title = str(rr.get('title', '') or '')
        body = str(rr.get('body', '') or '')
        evidence_text = f'{title} {body}'.strip()
        for cand in extract_agent_candidates(evidence_text):
            score = score_candidate(rr, cand['method'])
            candidate_rows.append({
                'transaction_id': rr.get('transaction_id'),
                'agent_name_raw': cand['agent_name'],
                'agent_name_norm': normalise_agent_name(cand['agent_name']).lower(),
                'extraction_method': cand['method'],
                'confidence_score': score,
                'confidence': confidence_label(score),
                'source_url': rr.get('href'),
                'source_domain': rr.get('domain'),
                'domain_class': rr.get('domain_class'),
                'source_title': rr.get('title'),
                'source_snippet': rr.get('body'),
                'query': rr.get('query'),
                'rank': rr.get('rank'),
                'created_utc': datetime.now(timezone.utc).isoformat(),
            })

candidates = pd.DataFrame(candidate_rows)
if len(candidates):
    candidates = candidates.sort_values(['transaction_id','confidence_score'], ascending=[True, False])

cand_path = OUT / 'agent_candidates_extracted_v3.csv'
candidates.to_csv(cand_path, index=False)
print('Candidate rows:', len(candidates))
display(candidates.head(50))


Candidate rows: 10


,transaction_id,agent_name_raw,agent_name_norm,extraction_method,confidence_score,confidence,source_url,source_domain,domain_class,source_title,source_snippet,query,rank,created_utc
7,{1A0C5C63-3EC0-7CBE-E063-4804A8C06C96},Q Estate Agents,q,known_agent_dictionary,73,medium_candidate,https://qestateagents.co.uk/property/denmark-t...,qestateagents.co.uk,possible_property_domain,"Denmark Terrace, Brighton, East Sussex, BN1 3A...","2 bedroom Flat to rent. Denmark Terrace, Brigh...","""DENMARK TERRACE"" ""BN1 3AN"" ""property for sale""",1.0,2026-04-27T19:25:22.088882+00:00
8,{1A0C5C63-3EC0-7CBE-E063-4804A8C06C96},"2 bedroom Flat to rent. Denmark Terrace, Brigh...","2 bedroom flat to rent. denmark terrace, brigh...",phrase_regex,63,medium_candidate,https://qestateagents.co.uk/property/denmark-t...,qestateagents.co.uk,possible_property_domain,"Denmark Terrace, Brighton, East Sussex, BN1 3A...","2 bedroom Flat to rent. Denmark Terrace, Brigh...","""DENMARK TERRACE"" ""BN1 3AN"" ""property for sale""",1.0,2026-04-27T19:25:22.089001+00:00
9,{34222872-371D-4D2B-E063-4704A8C07853},Savills,savills,known_agent_dictionary,63,medium_candidate,https://search.savills.com/property-detail/gbb...,search.savills.com,general_domain,"Davigdor Road, Hove, East Sussex, BN3 1RE | Ne...",Property details for Davigdor Road. One of man...,"""DAVIGDOR ROAD"" ""BN3 1RE"" ""property for sale""",1.0,2026-04-27T19:25:22.090948+00:00
0,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},Jacobs Steel,jacobs steel,known_agent_dictionary,98,high_candidate,https://www.rightmove.co.uk/properties/166579508,rightmove.co.uk,good_property_domain,"2 bedroom flat for sale in Windlesham Court, 4...","September 4, 2025 - 2 bedroom flat for sale in...","""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",8.0,2026-04-27T19:25:22.063087+00:00
1,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},"Jacobs Steel, West Worthing","jacobs steel, west worthing",phrase_regex,88,high_candidate,https://www.rightmove.co.uk/properties/166579508,rightmove.co.uk,good_property_domain,"2 bedroom flat for sale in Windlesham Court, 4...","September 4, 2025 - 2 bedroom flat for sale in...","""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",8.0,2026-04-27T19:25:22.063213+00:00
2,{C8A3A576-0FD8-0425-E053-6C04A8C0947B},in Eastbourne &,in eastbourne &,phrase_regex,45,low_candidate,https://www.eastbourne.co.uk/estate-agents/,eastbourne.co.uk,general_domain,Estate Agents in Eastbourne & Property Agents,0 Reviews 1.15 miles. Trident Properties Eastb...,"""BN21 1RS"" ""EASTBOURNE"" ""estate agent""",1.0,2026-04-27T19:25:22.066487+00:00
3,{C8A3A576-0FD8-0425-E053-6C04A8C0947B},Eastbourne.117 Green St. BN21 1RS Eastbourne E...,eastbourne.117 green st. bn21 1rs eastbourne e...,phrase_regex,45,low_candidate,https://eastbourne.cylex-uk.co.uk/houses+to+le...,eastbourne.cylex-uk.co.uk,general_domain,Houses to Let in Eastbourne Ask for free quotes,Estate Agents Eastbourne.117 Green St. BN21 1R...,"""BN21 1RS"" ""EASTBOURNE"" ""estate agent""",2.0,2026-04-27T19:25:22.066743+00:00
4,{C8A3A576-0FD8-0425-E053-6C04A8C0947B},opening times and... More,opening times and... more,phrase_regex,40,low_candidate,https://www.bigreddirectory.com/bn-lettings-ag...,bigreddirectory.com,general_domain,BN Lettings Agency Eastbourne Estate Agent ope...,More estate agents close to Eastbourne. Eastbo...,"""BN21 1RS"" ""EASTBOURNE"" ""estate agent""",3.0,2026-04-27T19:25:22.067117+00:00
6,{DE2D0CE0-2A0D-51EE-E053-6C04A8C00671},Aspire Residential,aspire residential,known_agent_dictionary,53,medium_candidate,https://www.aspireresidential.co.uk/find-a-pro...,aspireresidential.co.uk,general_domain,"Tavy Close, Worthing | Aspire Residential",Internally the property is well presented and ...,"""TAVY CLOSE"" ""BN13 3PA"" ""property for sale""",1.0,2026-04-27T19:25:22.073735+00:00
5,{F3B6C199-0687-6E40-E053-6C04A8C0B3B4},and get no-obligation quotes by entering your ...,and get no-obligation quotes by entering your ...,phrase_regex,70,medium_candidate,https://nethou

In [12]:
# ============================================================
# 11. Build canonical agent and transaction-agent tables
# ============================================================
def canonical_agent_id(name_norm):
    return hashlib.sha1(str(name_norm).encode('utf-8')).hexdigest()[:12]

if len(candidates):
    candidates['agent_id'] = candidates['agent_name_norm'].apply(canonical_agent_id)
    agents = (candidates
        .groupby(['agent_id','agent_name_norm'], as_index=False)
        .agg(
            agent_name_raw_example=('agent_name_raw','first'),
            evidence_rows=('agent_id','size'),
            max_confidence_score=('confidence_score','max'),
            first_seen_utc=('created_utc','min'),
            last_seen_utc=('created_utc','max')
        ))
    agent_transactions = (candidates
        .sort_values(['transaction_id','confidence_score'], ascending=[True, False])
        .drop_duplicates('transaction_id')
        .copy())
    agent_transactions = agent_transactions[[
        'transaction_id','agent_id','agent_name_raw','agent_name_norm','confidence_score','confidence',
        'extraction_method','source_url','source_domain','domain_class','source_title','source_snippet','query','created_utc'
    ]]
else:
    agents = pd.DataFrame(columns=['agent_id','agent_name_norm','agent_name_raw_example','evidence_rows','max_confidence_score'])
    agent_transactions = pd.DataFrame(columns=['transaction_id','agent_id','agent_name_raw','confidence_score','confidence'])

for name, df in {
    'agents_free_discovery_v3': agents,
    'agent_transactions_free_discovery_v3': agent_transactions,
}.items():
    df.to_csv(OUT / f'{name}.csv', index=False)
    try:
        df.to_parquet(OUT / f'{name}.parquet', index=False)
    except Exception as e:
        print(f'Parquet save warning for {name}:', e)

print('Agents:', len(agents))
print('Agent transaction links:', len(agent_transactions))
display(agents.head(50))
display(agent_transactions.head(50))


Agents: 10
Agent transaction links: 6


,agent_id,agent_name_norm,agent_name_raw_example,evidence_rows,max_confidence_score,first_seen_utc,last_seen_utc
0,22ea1c649c82,q,Q Estate Agents,1,73,2026-04-27T19:25:22.088882+00:00,2026-04-27T19:25:22.088882+00:00
1,3c37eda801f9,aspire residential,Aspire Residential,1,53,2026-04-27T19:25:22.073735+00:00,2026-04-27T19:25:22.073735+00:00
2,4bbc08c14b10,savills,Savills,1,63,2026-04-27T19:25:22.090948+00:00,2026-04-27T19:25:22.090948+00:00
3,4c98ceb8691d,opening times and... more,opening times and... More,1,40,2026-04-27T19:25:22.067117+00:00,2026-04-27T19:25:22.067117+00:00
4,5ab7c6b4e4dc,"2 bedroom flat to rent. denmark terrace, brigh...","2 bedroom Flat to rent. Denmark Terrace, Brigh...",1,63,2026-04-27T19:25:22.089001+00:00,2026-04-27T19:25:22.089001+00:00
5,94b849009c32,"jacobs steel, west worthing","Jacobs Steel, West Worthing",1,88,2026-04-27T19:25:22.063213+00:00,2026-04-27T19:25:22.063213+00:00
6,a9b5e193e239,jacobs steel,Jacobs Steel,1,98,2026-04-27T19:25:22.063087+00:00,2026-04-27T19:25:22.063087+00:00
7,aa0f75173db4,in eastbourne &,in Eastbourne &,1,45,2026-04-27T19:25:22.066487+00:00,2026-04-27T19:25:22.066487+00:00
8,adb77b1bb8e2,eastbourne.117 green st. bn21 1rs eastbourne e...,Eastbourne.117 Green St. BN21 1RS Eastbourne E...,1,45,2026-04-27T19:25:22.066743+00:00,2026-04-27T19:25:22.066743+00:00
9,f4bf82ebbeec,and get no-obligation quotes by entering your ...,and get no-obligation quotes by entering your ...,1,70,2026-04-27T19:25:22.070669+00:00,2026-04-27T19:25:22.070669+00:00


,transaction_id,agent_id,agent_name_raw,agent_name_norm,confidence_score,confidence,extraction_method,source_url,source_domain,domain_class,source_title,source_snippet,query,created_utc
7,{1A0C5C63-3EC0-7CBE-E063-4804A8C06C96},22ea1c649c82,Q Estate Agents,q,73,medium_candidate,known_agent_dictionary,https://qestateagents.co.uk/property/denmark-t...,qestateagents.co.uk,possible_property_domain,"Denmark Terrace, Brighton, East Sussex, BN1 3A...","2 bedroom Flat to rent. Denmark Terrace, Brigh...","""DENMARK TERRACE"" ""BN1 3AN"" ""property for sale""",2026-04-27T19:25:22.088882+00:00
9,{34222872-371D-4D2B-E063-4704A8C07853},4bbc08c14b10,Savills,savills,63,medium_candidate,known_agent_dictionary,https://search.savills.com/property-detail/gbb...,search.savills.com,general_domain,"Davigdor Road, Hove, East Sussex, BN3 1RE | Ne...",Property details for Davigdor Road. One of man...,"""DAVIGDOR ROAD"" ""BN3 1RE"" ""property for sale""",2026-04-27T19:25:22.090948+00:00
0,{C3C3F9B6-83F2-362B-E053-6B04A8C03ACC},a9b5e193e239,Jacobs Steel,jacobs steel,98,high_candidate,known_agent_dictionary,https://www.rightmove.co.uk/properties/166579508,rightmove.co.uk,good_property_domain,"2 bedroom flat for sale in Windlesham Court, 4...","September 4, 2025 - 2 bedroom flat for sale in...","""FLAT 9 WILMINGTON COURT ST VALERIE ROAD WORTH...",2026-04-27T19:25:22.063087+00:00
2,{C8A3A576-0FD8-0425-E053-6C04A8C0947B},aa0f75173db4,in Eastbourne &,in eastbourne &,45,low_candidate,phrase_regex,https://www.eastbourne.co.uk/estate-agents/,eastbourne.co.uk,general_domain,Estate Agents in Eastbourne & Property Agents,0 Reviews 1.15 miles. Trident Properties Eastb...,"""BN21 1RS"" ""EASTBOURNE"" ""estate agent""",2026-04-27T19:25:22.066487+00:00
6,{DE2D0CE0-2A0D-51EE-E053-6C04A8C00671},3c37eda801f9,Aspire Residential,aspire residential,53,medium_candidate,known_agent_dictionary,https://www.aspireresidential.co.uk/find-a-pro...,aspireresidential.co.uk,general_domain,"Tavy Close, Worthing | Aspire Residential",Internally the property is well presented and ...,"""TAVY CLOSE"" ""BN13 3PA"" ""property for sale""",2026-04-27T19:25:22.073735+00:00
5,{F3B6C199-0687-6E40-E053-6C04A8C0B3B4},f4bf82ebbeec,and get no-obligation quotes by entering your ...,and get no-obligation quotes by entering your ...,70,medium_candidate,phrase_regex,https://nethouseprices.com/house-prices/West+S...,nethouseprices.com,good_property_domain,"Sold House Prices in Hassocks, Marchants Road,...","44 Grand Avenue, Hassocks, BN6 8DE.Selling you...","""BN6 8HS"" ""HASSOCKS"" ""estate agent""",2026-04-27T19:25:22.070669+00:00


In [13]:
# ============================================================
# 12. Summaries and manifest
# ============================================================
sample_tx = pilot[['transaction_id','sale_year','postcode_district','address_clean','postcode_clean','price']].copy()
links_enriched = sample_tx.merge(agent_transactions, on='transaction_id', how='left')
links_enriched['has_candidate_agent'] = links_enriched['agent_id'].notna()

summary_by_year = (links_enriched
    .groupby('sale_year', dropna=False)
    .agg(sample_rows=('transaction_id','size'), candidate_rows=('has_candidate_agent','sum'))
    .reset_index())
summary_by_year['hit_rate'] = summary_by_year['candidate_rows'] / summary_by_year['sample_rows']

summary_by_district = (links_enriched
    .groupby('postcode_district', dropna=False)
    .agg(sample_rows=('transaction_id','size'), candidate_rows=('has_candidate_agent','sum'))
    .reset_index())
summary_by_district['hit_rate'] = summary_by_district['candidate_rows'] / summary_by_district['sample_rows']

if len(agent_transactions):
    summary_by_agent = (agent_transactions
        .groupby(['agent_id','agent_name_norm'], as_index=False)
        .agg(transaction_links=('transaction_id','nunique'), max_confidence_score=('confidence_score','max'))
        .sort_values('transaction_links', ascending=False))
else:
    summary_by_agent = pd.DataFrame(columns=['agent_id','agent_name_norm','transaction_links','max_confidence_score'])

summary_by_year.to_csv(OUT / 'summary_by_year_v3.csv', index=False)
summary_by_district.to_csv(OUT / 'summary_by_postcode_district_v3.csv', index=False)
summary_by_agent.to_csv(OUT / 'summary_by_agent_v3.csv', index=False)
links_enriched.to_csv(OUT / 'sample_transactions_with_agent_candidates_v3.csv', index=False)

manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'postcode_prefix': POSTCODE_PREFIX,
    'years': YEARS,
    'transactions_sampled': int(len(pilot)),
    'query_plan_rows': int(len(query_plan)),
    'raw_search_result_rows': int(len(raw)) if 'raw' in globals() else 0,
    'candidate_rows': int(len(candidates)),
    'unique_agents': int(len(agents)),
    'transactions_with_candidate_agent': int(agent_transactions['transaction_id'].nunique()) if len(agent_transactions) else 0,
    'hit_rate_transactions': float(agent_transactions['transaction_id'].nunique() / len(pilot)) if len(pilot) else 0.0,
    'search_backend': SEARCH_BACKEND,
    'max_transactions_this_run': MAX_TRANSACTIONS_THIS_RUN,
    'max_results_per_query': MAX_RESULTS_PER_QUERY,
    'sleep_seconds_between_queries': SLEEP_SECONDS_BETWEEN_QUERIES,
    'portal_only_mode': PORTAL_ONLY_MODE,
    'keep_only_useful_domains': KEEP_ONLY_USEFUL_DOMAINS,
}
with open(OUT / 'agent_attribution_free_v3_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print('Summary by year')
display(summary_by_year)
print('Summary by district')
display(summary_by_district.sort_values(['candidate_rows','sample_rows'], ascending=False).head(30))
print('Summary by agent')
display(summary_by_agent.head(30))


{
  "created_utc": "2026-04-27T19:25:22.453186+00:00",
  "postcode_prefix": "BN",
  "years": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "transactions_sampled": 25,
  "query_plan_rows": 200,
  "raw_search_result_rows": 308,
  "candidate_rows": 10,
  "unique_agents": 10,
  "transactions_with_candidate_agent": 6,
  "hit_rate_transactions": 0.24,
  "search_backend": "ddgs",
  "max_transactions_this_run": 25,
  "max_results_per_query": 8,
  "sleep_seconds_between_queries": 8,
  "portal_only_mode": false,
  "keep_only_useful_domains": false
}
Summary by year


,sale_year,sample_rows,candidate_rows,hit_rate
0,2021,5,3,0.6
1,2022,5,1,0.2
2,2023,5,0,0.0
3,2024,5,2,0.4
4,2025,5,0,0.0


Summary by district


,postcode_district,sample_rows,candidate_rows,hit_rate
14,BN3,3,1,0.333333
0,BN1,2,1,0.500000
1,BN11,1,1,1.000000
3,BN13,1,1,1.000000
8,BN21,1,1,1.000000
15,BN6,1,1,1.000000
6,BN2,3,0,0.000000
5,BN18,2,0,0.000000
7,BN20,2,0,0.000000
2,BN12,1,0,0.000000


Summary by agent


,agent_id,agent_name_norm,transaction_links,max_confidence_score
0,22ea1c649c82,q,1,73
1,3c37eda801f9,aspire residential,1,53
2,4bbc08c14b10,savills,1,63
3,a9b5e193e239,jacobs steel,1,98
4,aa0f75173db4,in eastbourne &,1,45
5,f4bf82ebbeec,and get no-obligation quotes by entering your ...,1,70


## Interpretation guidance

Expected results are still probabilistic. If v3 improves from the earlier 4% hit rate to roughly 10–20%, it is working better. If it remains below 5%, free DDG-style discovery is probably too weak as the main acquisition method.

Recommended next tuning after this run:
1. Inspect `agent_discovery_raw_search_results_v3_classified.csv` for useful domains.
2. Add newly discovered local agent names to `KNOWN_AGENTS`.
3. Try `PORTAL_ONLY_MODE=True` for precision testing.
4. Try `KEEP_ONLY_USEFUL_DOMAINS=True` if there are too many false positives.
5. Increase `MAX_TRANSACTIONS_THIS_RUN` only after candidate quality is acceptable.
